In [1]:
!pip install pymongo pandas matplotlib requests beautifulsoup4 kafka-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 40.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 30.0 MB/s eta 0:00:00


In [7]:
import pandas as pd
from pymongo import MongoClient
import numpy as np

# 1. Leer el CSV asegurando que NO use la primera columna como índice
# Esto evita que los nombres se desplacen
df = pd.read_csv('steam_kaggle.csv', index_col=False, low_memory=False)

# 2. Limpiar nombres de columnas
df.columns = df.columns.str.strip()

# 3. Seleccionar columnas por nombre real
cols_interes = ['Name', 'Release date', 'Price', 'Positive', 'Negative', 'Genres']
df_limpio = df[cols_interes].copy()

# 4. Limpieza de datos
# Convertir fechas (los errores se vuelven NaT)
df_limpio['Release date'] = pd.to_datetime(df_limpio['Release date'], errors='coerce')

# Convertir precios y votos a números
df_limpio['Price'] = pd.to_numeric(df_limpio['Price'], errors='coerce').fillna(0)
df_limpio['Positive'] = pd.to_numeric(df_limpio['Positive'], errors='coerce').fillna(0)
df_limpio['Negative'] = pd.to_numeric(df_limpio['Negative'], errors='coerce').fillna(0)

# 5. EL TRUCO PARA MONGODB: Convertir NaT/NaN a None
# MongoDB no acepta NaT de Pandas, pero acepta None de Python
df_mongo = df_limpio.replace({np.nan: None})
# Para las fechas específicamente:
df_mongo['Release date'] = df_mongo['Release date'].where(df_mongo['Release date'].notnull(), None)

print("\n--- COMPROBACIÓN FINAL ---")
print(df_mongo[['Name', 'Release date']].head(5))

# 6. Guardar en MongoDB
client = MongoClient('mongodb://mongodb:27017')
db = client['steam_db']
db.juegos_historicos.drop() 

documentos = df_mongo.to_dict('records')
db.juegos_historicos.insert_many(documentos)

print(f"\n{len(documentos)} juegos cargados sin errores de formato.")


--- COMPROBACIÓN FINAL ---
                                    Name Release date
0             Black Dragon Mage Playtest   2023-08-01
1  Supipara - Chapter 1 Spring Has Come!   2016-07-29
2      Mystery Solitaire The Black Raven   2019-05-06
3            버튜버 파라노이아 - Vtuber Paranoia   2024-10-31
4                          Maze Quest VR   2025-04-24

122611 juegos cargados sin errores de formato.
